# Fine-tune IndicTrans2 for English ↔ Punjabi (LoRA)

LoRA fine-tunes AI4Bharat's IndicTrans2 specifically on the English↔Punjabi pair using the [Samanantar](https://huggingface.co/datasets/ai4bharat/samanantar) parallel corpus. The stock checkpoints split capacity across 22 Indic languages; specializing just for Punjabi can sharpen accuracy without a full retrain (LoRA only trains a small adapter on top of the frozen base model).

**Runtime > Change runtime type > select a GPU (T4 is fine on the free tier)** before running any cells below.

### Prerequisites
1. Create a (free) Hugging Face account.
2. Accept the license/access request on the model page(s) you intend to train:
   - https://huggingface.co/ai4bharat/indictrans2-en-indic-1B
   - https://huggingface.co/ai4bharat/indictrans2-indic-en-1B (only needed for the pa→en direction)
3. Log in below so the gated model can be downloaded.

In [ ]:
!pip install -q transformers peft accelerate datasets sentencepiece IndicTransToolkit huggingface_hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch
from datasets import load_dataset
from IndicTransToolkit.processor import IndicProcessor  # type: ignore[import-not-found]
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

assert torch.cuda.is_available(), "No GPU detected - set Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))

## Configuration

Set `DIRECTION` to `"en-pa"` (English→Punjabi) or `"pa-en"` (Punjabi→English). Run this whole notebook once per direction if you want both.

In [ ]:
DIRECTION = "en-pa"  # or "pa-en"
EPOCHS = 2.0
BATCH_SIZE = 8

# Samanantar has ~700K English<->Punjabi pairs total; these caps keep a single
# training run within a reasonable time/VRAM budget on a free Colab GPU.
# Raise them if you have more time/compute available.
NUM_TRAIN_EXAMPLES = 200_000
NUM_EVAL_EXAMPLES = 2_000
MAX_LENGTH = 256
OUTPUT_DIR = f"./indictrans2-{DIRECTION}-punjabi-lora"

LANG_CODES = {"en": "eng_Latn", "pa": "pan_Guru"}
BASE_MODEL = {
    "en-pa": "ai4bharat/indictrans2-en-indic-1B",
    "pa-en": "ai4bharat/indictrans2-indic-en-1B",
}[DIRECTION]
print("Base model:", BASE_MODEL)

## Load and preprocess the Samanantar Punjabi subset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
ip = IndicProcessor(inference=True)

src_lang_code = LANG_CODES["en" if DIRECTION == "en-pa" else "pa"]
tgt_lang_code = LANG_CODES["pa" if DIRECTION == "en-pa" else "en"]

# Samanantar's Punjabi config always has 'src' = English, 'tgt' = Punjabi,
# regardless of which direction we're training - we swap below as needed.
raw = load_dataset("ai4bharat/samanantar", "pa", split="train")
raw = raw.shuffle(seed=42)
total = min(NUM_TRAIN_EXAMPLES + NUM_EVAL_EXAMPLES, len(raw))
raw = raw.select(range(total))
split = raw.train_test_split(test_size=NUM_EVAL_EXAMPLES, seed=42)


def preprocess(batch):
    sources = batch["src"] if DIRECTION == "en-pa" else batch["tgt"]
    targets = batch["tgt"] if DIRECTION == "en-pa" else batch["src"]

    processed_sources = ip.preprocess_batch(sources, src_lang=src_lang_code, tgt_lang=tgt_lang_code)
    model_inputs = tokenizer(processed_sources, truncation=True, max_length=MAX_LENGTH)

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, truncation=True, max_length=MAX_LENGTH)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


dataset = split.map(preprocess, batched=True, remove_columns=split["train"].column_names)
dataset

## Load the base model and wrap it with a LoRA adapter

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL, trust_remote_code=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",  # robust to IndicTrans2's custom module names
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Train

In [ ]:
collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=EPOCHS,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    fp16=True,
    predict_with_generate=True,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=collator,
    tokenizer=tokenizer,
)

trainer.train()

## Save the LoRA adapter

This saves only the small adapter weights, not the full base model. Download the resulting folder (e.g. zip it) before your Colab session ends, since Colab storage isn't persistent.

In [ ]:
final_dir = OUTPUT_DIR + "-final"
trainer.save_model(final_dir)
print(f"Saved LoRA adapter to {final_dir}")
print(
    "Load it later with:\n"
    "  from peft import PeftModel\n"
    "  base = AutoModelForSeq2SeqLM.from_pretrained(" + repr(BASE_MODEL) + ", trust_remote_code=True)\n"
    "  model = PeftModel.from_pretrained(base, " + repr(final_dir) + ")"
)